In [ ]:
## TODO: All project members will need to pip install rasterio 

import torch
import numpy as np
import pandas as pd
import rasterio
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader, Subset

DIR_PATH = "../EuroSATallBands/"

#dataloader class
class EuroSATDataset(Dataset):
    def __init__(self, csv_file):
        df = pd.read_csv(DIR_PATH + csv_file)

        self.files = df.iloc[:, 0].tolist()
        #make sure the class IDs are parsed as an int
        self.labels = df.iloc[:, 1].astype(int).tolist()

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        filename = DIR_PATH + self.files[index]
        #must use rasterio to import the satellite TIF
        with rasterio.open(filename) as img:
            image = img.read().astype(np.float32)

        X = torch.from_numpy(image)
        #this is needed for CrossEntropyLoss later
        Y = torch.tensor(self.labels[index], dtype = torch.int)

        return X, Y

#default parameters: batch size 32, no max, do not shuffle, no workers, random seed of 2 for consistency
def make_dataloader(csv_file, batchsize=32, max_samples = None, shuffle = False, numWorkers = 0, seed = 2):
    dataset = EuroSATDataset(csv_file)
    #This is for splitting the data into smaller subsets so we don't have to train on the whole dataset
    if max_samples is not None:
        if max_samples < 1:
            raise ValueError("Maximum number of records must be positive")

        #for consistency in comparing models
        rng = torch.Generator().manual_seed(seed)
        indices = torch.randperm(dataset.__len__(), generator = rng)[:max_samples].tolist()

        dataset = Subset(dataset, indices)

    return DataLoader(dataset,
                      batch_size = batchsize, 
                      shuffle = shuffle, 
                      num_workers = numWorkers, 
                      pin_memory = torch.cuda.is_available(), 
                      persistent_workers = numWorkers > 0)


In [35]:
# small dataset for model testing
train_loader = make_dataloader(
    "train.csv",
    batchsize=32,
    max_samples=500,
    shuffle=True
)

valid_loader = make_dataloader(
    "validation.csv",
    batchsize=32,
    max_samples=100
)

test_loader = make_dataloader(
    "test.csv",
    batchsize=32,
)
# full dataset
# train_loader = make_dataloader(
#     "train.csv",
#     batchsize=32,
#     shuffle=True
# )

# valid_loader = make_dataloader(
#     "validation.csv",
#     batchsize=32,
# )

# test_loader = make_dataloader(
#     "test.csv",
#     batchsize=32,
# )

In [36]:
#TQDM is here for a built in training progress bar. Put model training below here. 
for X, Y in tqdm(train_loader, desc="Training", unit="batch"):
    pass

Training:   0%|          | 0/16 [00:00<?, ?batch/s]

In [100]:
X, Y = next(iter(train_loader))

print("Images:", X.shape)
print("Labels:", Y.shape)
print("Data type:", X.dtype)

Images: torch.Size([32, 13, 64, 64])
Labels: torch.Size([32])
Data type: torch.float32
